In [1]:
# function to calculate the autopet3_py metrics 

# Method to get the lesion metrics and save as a csv

In [9]:

import os
import glob
import json
import pickle

import numpy as np
import pandas as pd
import nibabel as nib
import SimpleITK as sitk

import numpy as np
import nibabel as nib
import pathlib as plb
import cc3d
import csv
import sys

import argparse

def handle_arguments():
    """
    Handle command line arguments for processing images.
    """
    parser = argparse.ArgumentParser(description='Process probability analysis with various options')
    parser.add_argument('-k', '--key', type=str, default='*', 
                       help='Key pattern for file matching (default: *)')
    parser.add_argument('-l', '--lesion-only', action='store_true', 
                       help='Process only lesion data, skip CTImage object creation')
    parser.add_argument('-f', '--fold', type=str, default='comb', 
                       help='Fold name/number (default: 1)')
    parser.add_argument('-s', '--skip-existing', action='store_true',
                       help='Skip processing files that have already been processed', default=False)
    parser.add_argument('-i', '--image-dir', type=str, default="//dartfs/rc/lab/B/BhattacharyaI/Public_Datasets/Autopet_III_nnunet_raw/Dataset888_AutoPet/imagesTr/", help='Directory containing images (default: ../imagesTr/)')
    parser.add_argument('-m', '--mets-only', action='store_true', 
                       help='Process only mets data, skip lesion-only data')
    parser.add_argument('-ld', '--label-dir', type=str, default="//dartfs/rc/lab/B/BhattacharyaI/Results/", help='Directory containing labels (default: ../labelsTr/)') 
    parser.add_argument('-b', '--bone-dir', type=str, default="//dartfs/rc/lab/B/BhattacharyaI/Results/Biratal/bone-metastasis/totalsegmentator/bone_segmentations_fold1/", help='Directory containing bone segmentations (default: ../totalsegmentator/bone_segmentations_fold1/)')

    parser.add_argument('-p', '--pred-dir', type=str, default="//dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset888_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_1_ens_comb/test_predictions/", 
                        help='Directory containing prediction files (default: //dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset888_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_1_ens_comb/test_predictions/)')

    parser.add_argument('-u', '--uncertainty-dir', type=str, default="//dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset888_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_1_ens_comb/uncertainty_maps/", 
                        help='Directory containing uncertainty maps (default: //dartfs/rc/lab/B/BhattacharyaI/Results/nnUNet_data/nnUNet_results/Dataset888_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_1_ens_comb/uncertainty_maps/)')
    parser.add_argument('-o', '--save-dir', type=str, default=args.pred_dir, 
                        help='Directory to save lesion analysis results (default: ./lesion_analysis_fold)')
    args = parser.parse_args()
    return args


def nii2numpy(nii_path):
    # input: path of NIfTI segmentation file, output: corresponding numpy array and voxel_vol in ml
    mask_nii = nib.load(str(nii_path))
    mask = mask_nii.get_fdata()
    pixdim = mask_nii.header['pixdim']   
    voxel_vol = pixdim[1]*pixdim[2]*pixdim[3]/1000
    return mask, voxel_vol


def con_comp(seg_array):
    # input: a binary segmentation array output: an array with seperated (indexed) connected components of the segmentation array
    connectivity = 18
    conn_comp = cc3d.connected_components(seg_array, connectivity=connectivity)
    return conn_comp



def false_pos_pix(gt_array,pred_array):
    # compute number of voxels of false positive connected components in prediction mask
    pred_conn_comp = con_comp(pred_array)
    
    false_pos = 0
    false_detected_lesion_count = 0
    
    for idx in range(1,pred_conn_comp.max()+1):
        comp_mask = np.isin(pred_conn_comp, idx)
        if (comp_mask*gt_array).sum() == 0:
            false_pos = false_pos+comp_mask.sum()
            false_detected_lesion_count += 1
    return false_pos, false_detected_lesion_count, pred_conn_comp.max()

def false_neg_pix(gt_array,pred_array):
    # compute number of voxels of false negative connected components (of the ground truth mask) in the prediction mask
    gt_conn_comp = con_comp(gt_array)
    
    false_neg = 0
    false_missed_lesion_count = 0
    
    for idx in range(1,gt_conn_comp.max()+1):
        comp_mask = np.isin(gt_conn_comp, idx)
        if (comp_mask*pred_array).sum() == 0:
            false_neg = false_neg+comp_mask.sum()
            false_missed_lesion_count += 1
            
    return false_neg, false_missed_lesion_count, gt_conn_comp.max()

def dice_score(mask1,mask2):
    # compute foreground Dice coefficient
    overlap = (mask1*mask2).sum()
    sum = mask1.sum()+mask2.sum()
    dice_score = 2*overlap/sum
    return dice_score



def compute_metrics(nii_gt_path, nii_pred_path):
    # main function
    gt_array, voxel_vol = nii2numpy(nii_gt_path)
    pred_array, voxel_vol = nii2numpy(nii_pred_path)
    false_neg_vol, false_missed_lesion_count, gt_lesion_count= false_neg_pix(gt_array, pred_array)
    false_neg_vol = false_neg_vol*voxel_vol

    false_pos_vol, false_detected_lesion_count, pred_lesion_count = false_pos_pix(gt_array, pred_array)
    false_pos_vol = false_pos_vol*voxel_vol
    
    dice_sc = dice_score(gt_array,pred_array)

    return dice_sc, false_pos_vol, false_neg_vol, false_missed_lesion_count, false_detected_lesion_count

def convert_location_to_case_name(file_locations):
    case_names = []
    for path in file_locations:
        parts = path.split('/')
        # Extract patient ID from PETCT_xxxx
        patient_folder = parts[2]  # e.g., 'PETCT_0011f3deaf'
        patient_id = patient_folder.replace('PETCT_', '')
        
        # Extract scan folder (date-NA-scan_type-number)
        scan_folder = parts[3]  # e.g., '03-23-2003-NA-PET-CT Ganzkoerper...-10445'
        
        # Create CSV filename
        case_name = f"fdg_{patient_id}_{scan_folder}"
        case_names.append(case_name)
    
    return np.array(case_names)




In [10]:
import tqdm
nnUNet_results = os.environ['nnUNet_results']
dataset_name = "Dataset999_AutoPet"
fold = "fold_1"
testing_split = "test_predictions"

pred_dir="{nnUNet_results}/{dataset_name}/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/{fold}/{testing_split}"
gt_dir = "{nnUNet_results}/test/gt/"

out_dir="{nnUNet_results}/{dataset_name}/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/{fold}/{testing_split}/lesion_metrics.csv"


In [11]:
prediction_file_names = [x for x in os.listdir(pred_dir.format(nnUNet_results=nnUNet_results,
 dataset_name=dataset_name, 
 fold=fold, 
 testing_split=testing_split)) if x.endswith('.nii.gz')]

gt_files = [os.path.join(gt_dir.format(nnUNet_results=nnUNet_results), x) for x in prediction_file_names]  # Match GT files to prediction files based on filename
prediction_files = [os.path.join(pred_dir.format(nnUNet_results=nnUNet_results,
 dataset_name=dataset_name,
 fold=fold,
 testing_split=testing_split), x) for x in prediction_file_names]
 

In [12]:
def read_summary_file(path, fold, validation): 
    with open(path.format(fold=fold, validation=validation), "r") as f:
        return json.load(f)
    
def load_dataset_folds(dataset_num, folds=[f'fold_{i}' for i in range(0, 11)], validation="validation_279"):
    path = DEFAULT_SUMMARY_PATH.format(
        input_dataset=f"Dataset{dataset_num}_AutoPet",
        fold="{fold}",
        validation="{validation}"
    )
    

    summary_dfs = []
    fdg_dfs = []
    psma_dfs = []
    
    for i in folds:
        filepath = path.format(fold=i, validation=validation)
        try:
            fold_data = read_summary_file(path, fold=i, validation=validation)
            print(f"Successfully read fold {i} for dataset {dataset_num}")
            # --- Per-case metrics, split by tracer ---
            fdg_rows = []
            psma_rows = []
            lesion_metrics_path = filepath.replace("summary.json", "lesion_metrics.csv")
            lesion_metrics_df = pd.read_csv(lesion_metrics_path)

            for case in fold_data['metric_per_case']:
                metrics = case['metrics']['1']
                pred_file = case['prediction_file']
                case_name = os.path.basename(pred_file).replace('.nii.gz', '')
                
                # now for this case, find the actual FP Vol from the lesion_metrics file
                # get for that case
                lesion_subdf = lesion_metrics_df[lesion_metrics_df['filename'] == case_name]

                # only add to dice if 

                row = {
                    "Dice": metrics['Dice'],
                    "FP":   lesion_subdf['false_pos_vol'].values[0] if not lesion_subdf.empty else np.nan,
                    "FN":   lesion_subdf['false_neg_vol'].values[0] if not lesion_subdf.empty else np.nan,
                }
                
                if case_name.lower().startswith('fdg'):
                    fdg_rows.append(row)
                elif case_name.lower().startswith('psma'):
                    psma_rows.append(row)
            
            def make_summary_row(rows, fold_label):
                if not rows:
                    return None
                df = pd.DataFrame(rows)
                return pd.DataFrame({
                    "fold":          [fold_label],
                    "Dice":          [round(df['Dice'].mean(), 4)],
                    "FP (voxels)":   [int(df['FP'].mean())],
                    "FN (voxels)":   [int(df['FN'].mean())],
                    "n_cases":       [len(rows)],
                })
            
            # Overall summary from foreground_mean
            df_summary = pd.DataFrame({
                "fold":        [str(i)],
                "Dice":        [round(fold_data['foreground_mean']['Dice'], 4)],
                "FP (voxels)": [int(fold_data['foreground_mean']['FP'])],
                "FN (voxels)": [int(fold_data['foreground_mean']['FN'])],
                "n_cases":     [len(fold_data['metric_per_case'])],
            })
            summary_dfs.append(df_summary)
            
            row_fdg  = make_summary_row(fdg_rows,  str(i))
            row_psma = make_summary_row(psma_rows, str(i))
            if row_fdg  is not None: fdg_dfs.append(row_fdg)
            if row_psma is not None: psma_dfs.append(row_psma)
            
        except Exception as e:
            print(f"Could not read fold {i} for dataset {dataset_num}")
            print(f"Expected path: {filepath}")
            print(f"Error: {e}")
    
    summary_df = pd.concat(summary_dfs, ignore_index=True).sort_values('fold')
    fdg_df     = pd.concat(fdg_dfs,     ignore_index=True).sort_values('fold') if fdg_dfs  else None
    psma_df    = pd.concat(psma_dfs,    ignore_index=True).sort_values('fold') if psma_dfs else None
    
    return summary_df, fdg_df, psma_df


In [13]:
import os

DEFAULT_SUMMARY_PATH = "{nnUNet_results}/{input_dataset}/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/{fold}/{validation}/summary.json"\
    .format(nnUNet_results=os.environ['nnUNet_results'], input_dataset="{input_dataset}", fold="{fold}", validation="{validation}") 
    
print(DEFAULT_SUMMARY_PATH)

/scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/{input_dataset}/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/{fold}/{validation}/summary.json


In [14]:
load_dataset_folds(dataset_num=999, folds=[f'fold_{i}' for i in range(0, 11)], validation="validation_279")

Successfully read fold fold_0 for dataset 999
Could not read fold fold_0 for dataset 999
Expected path: /scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset999_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_0/validation_279/summary.json
Error: cannot convert float NaN to integer
Successfully read fold fold_1 for dataset 999
Could not read fold fold_1 for dataset 999
Expected path: /scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset999_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_1/validation_279/summary.json
Error: cannot convert float NaN to integer
Successfully read fold fold_2 for dataset 999
Could not read fold fold_2 for dataset 999
Expected path: /scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset999_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_2/validation_279/summary.json
Error: cannot conver

(       fold    Dice  FP (voxels)  FN (voxels)  n_cases
 0    fold_0  0.3215          430         7807      247
 1    fold_1  0.5730         2049         3480      247
 10  fold_10  0.6679         2231         2286      247
 2    fold_2  0.5937         2501         3265      247
 3    fold_3  0.6167         2495         2767      247
 4    fold_4  0.6090         1726         3087      247
 5    fold_5  0.6348         1853         3256      247
 6    fold_6  0.6295         1758         2977      247
 7    fold_7  0.6347         2084         2876      247
 8    fold_8  0.6523         2213         2464      247
 9    fold_9  0.6587         2444         1970      247,
 None,
 None)

## Once you have the lesion metrics, processing to generate the tables 
This is to generate the figures for the validation table figures. So the Dice, FP Vol, FN Vol across disease 

In [18]:
def load_dataset_folds_with_diagnosis(
    dataset_num, 
    folds=[f'fold_{i}' for i in range(0, 11)], 
    validation="validation_279"
):
    path = DEFAULT_SUMMARY_PATH.format(
        input_dataset=f"Dataset{dataset_num}_AutoPet",
        fold="{fold}",
        validation="{validation}"
    )

    # Load FDG metadata once upfront
    fdg_metadata = pd.read_csv('fdg_metadata.csv')
    fdg_metadata['file'] = convert_location_to_case_name(fdg_metadata['File Location'].values)
    fdg_metadata = fdg_metadata[['file', 'diagnosis']].drop_duplicates(subset=['file', 'diagnosis'])

    def get_diagnosis(case_name):
        """Look up diagnosis from metadata, falling back to PROSTATE_CANCER for PSMA."""
        if case_name.lower().startswith('psma'):
            return 'PROSTATE_CANCER'
        match = fdg_metadata[fdg_metadata['file'] == case_name]
        if not match.empty:
            return match['diagnosis'].values[0]
        return None



    summary_dfs = []
    # Store individual rows across all folds, we'll aggregate at the end
    all_rows = []

    for fold in folds:
        filepath = path.format(fold=fold, validation=validation)
        try:
            fold_data = read_summary_file(path, fold=fold, validation=validation)

            lesion_metrics_path = filepath.replace("summary.json", "lesion_metrics.csv")
            lesion_metrics_df = pd.read_csv(lesion_metrics_path)
            # Strip extension once
            lesion_metrics_df['filename'] = lesion_metrics_df['filename'].str.replace('.nii.gz', '', regex=False)

            for case in fold_data['metric_per_case']:
                metrics = case['metrics']['1']
                pred_file = case['prediction_file']
                case_name = os.path.basename(pred_file).replace('.nii.gz', '')

                lesion_subdf = lesion_metrics_df[lesion_metrics_df['filename'] == case_name]

                tracer = None
                if case_name.lower().startswith('fdg'):
                    tracer = 'FDG'
                elif case_name.lower().startswith('psma'):
                    tracer = 'PSMA'

                diagnosis = get_diagnosis(case_name)

                all_rows.append({
                    "fold":      fold,
                    "fold_num":  int(fold.split('_')[1]),
                    "case_name": case_name,
                    "tracer":    tracer,
                    "diagnosis": diagnosis,
                    "Dice":      metrics['Dice'],
                    "false_pos_vol":        lesion_subdf['false_pos_vol'].values[0] if not lesion_subdf.empty else np.nan,
                    "false_neg_vol":        lesion_subdf['false_neg_vol'].values[0] if not lesion_subdf.empty else np.nan,
                })

            fold_rows = [r for r in all_rows if r['fold'] == fold]
            fold_df = pd.DataFrame(fold_rows)

            summary_dfs.append(pd.DataFrame({
                "fold":        [str(fold)],
                "fold_num":    [int(fold.split('_')[1])],
                "Dice":        [round(fold_data['foreground_mean']['Dice'], 4)],
                "false_pos_vol": [round(fold_df['false_pos_vol'].mean(), 2)],
                "false_neg_vol": [round(fold_df['false_neg_vol'].mean(), 2)],
                "n_cases":     [len(fold_data['metric_per_case'])],
            }))
        except Exception as e:
            print(f"Could not read fold {fold} for dataset {dataset_num}")
            print(f"Expected path: {filepath}")
            print(f"Error: {e}")

    # Build a flat dataframe of all cases
    all_cases_df = pd.DataFrame(all_rows)

    def make_summary_table(df, group_col):
        """Aggregate metrics grouped by fold + some grouping column (tracer or diagnosis)."""
        return (

            df.groupby(['fold_num', group_col])
            .agg(
                Dice=('Dice', 'mean'),
                false_pos_vol=('false_pos_vol', 'mean'),
                false_neg_vol=('false_neg_vol', 'mean'),
                n_cases=('case_name', 'count'),
            )
            .round({'Dice': 4, 'false_pos_vol': 0, 'false_neg_vol': 0})
            .reset_index()
            .rename(columns={'false_pos_vol': 'FP (ml)', 'false_neg_vol': 'FN (ml)'})
            .sort_values(['fold_num', group_col])
        )

    summary_df   = pd.concat(summary_dfs, ignore_index=True).sort_values('fold_num')
    tracer_df    = make_summary_table(all_cases_df, 'tracer')
    diagnosis_df = make_summary_table(all_cases_df, 'diagnosis')



    return summary_df, tracer_df, diagnosis_df, all_cases_df


summary, tracer, diagnosis, all_cases = load_dataset_folds_with_diagnosis(dataset_num=111, folds=[f'fold_{i}' for i in range(0, 12)], 
validation="validation_279")

Could not read fold fold_9 for dataset 111
Expected path: /scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset111_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_9/validation_279/summary.json
Error: [Errno 2] No such file or directory: '/scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset111_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_9/validation_279/summary.json'
Could not read fold fold_10 for dataset 111
Expected path: /scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset111_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_10/validation_279/summary.json
Error: [Errno 2] No such file or directory: '/scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset111_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_10/validation_279/summary.json'
Could not rea

In [19]:
all_cases['diseased'] = all_cases['diagnosis'] != 'NEGATIVE'
all_cases['diseased'].value_counts()

diseased
True     1557
False     666
Name: count, dtype: int64

In [20]:
all_cases.groupby(['diseased', 'tracer', 'fold_num']).agg({'Dice': 'mean', 'false_pos_vol': 'mean', 'false_neg_vol': 'mean', 'case_name': 'count'}).round(3)


Dice  false_pos_vol  false_neg_vol  case_name
diseased tracer fold_num                                                
False    FDG    0           NaN         37.613          0.000         74
                1           NaN         32.099          0.000         74
                2           NaN         30.575          0.000         74
                3           NaN         35.928          0.000         74
                4           NaN         32.250          0.000         74
                5           NaN         34.495          0.000         74
                6           NaN         36.492          0.000         74
                7           NaN         35.096          0.000         74
                8           NaN         42.112          0.000         74
True     FDG    0         0.663          4.190         34.830         77
                1         0.690          2.991         18.831         77
                2         0.715          3.573         30.946         77
                3         0.726          4.487         12.577         77
                4         0.717          3.408         13.686         77
                5         0.742          3.444         12.796         77
                6         0.735          3.200         13.311         77
                7         0.750          4.051          9.182         77
                8         0.753          4.005         11.170         77
         PSMA   0         0.530         20.071         11.850         96
                1         0.545         19.729         14.477         96
                2         0.552         20.054         11.120         96
                3         0.556         20.322         11.633         96
                4         0.579         21.233          7.896         96
                5         0.565         20.937          9.527         96
                6         0.570         20.837         10.204         96
                7         0.579         20.205         10.269         96
                8         0.586         19.719          9.500         96

In [ ]:
all_cases

,fold,fold_num,case_name,tracer,diagnosis,Dice,false_pos_vol,false_neg_vol
0,fold_0,0,fdg_0011f3deaf_03-23-2003-NA-PET-CT Ganzkoerpe...,FDG,MELANOMA,0.034310,0.945518,14.879462
1,fold_0,0,fdg_01140d52d8_08-13-2005-NA-PET-CT Ganzkoerpe...,FDG,MELANOMA,0.939595,4.814675,0.124410
2,fold_0,0,fdg_0117d7f11f_09-13-2001-NA-PET-CT Ganzkoerpe...,FDG,LUNG_CANCER,0.005514,0.186615,178.242513
3,fold_0,0,fdg_0410759456_08-22-2003-NA-PET-CT Ganzkoerpe...,FDG,NEGATIVE,NaN,2.214502,0.000000
4,fold_0,0,fdg_0410759456_11-07-2002-NA-PET-CT Ganzkoerpe...,FDG,NEGATIVE,NaN,4.740029,0.000000
...,...,...,...,...,...,...,...,...
2959,fold_11,11,psma_fa6ce43d309315b9_2018-03-03,PSMA,PROSTATE_CANCER,0.656357,0.146695,2.738306
2960,fold_11,11,psma_fbd11b7e8c246d80_2018-10-05,PSMA,PROSTATE_CANCER,0.800277,0.317839,0.000000
2961,fold_11,11,psma_fcae072067609bd2_2014-07-26,PSMA,PROSTATE_CANCER,0.344828,0.000000,7.530341
2962,fold_11,11,psma_fe15e4b0571c9233_2018-02-18,PSMA,PROSTATE_CANCER,0.646550,16.986057,13.734820


In [ ]:
summary, tracer, diagnosis, all_cases = load_dataset_folds_with_diagnosis(dataset_num=111, folds=[f'fold_{i}' for i in range(0, 11)], 
validation="validation_279")
summary

Successfully read fold fold_0 for dataset 111
Could not read fold fold_0 for dataset 111
Expected path: /scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset111_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_0/test_predictions/summary.json
Error: [Errno 2] No such file or directory: '/scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset111_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_0/test_predictions/lesion_metrics.csv'
Successfully read fold fold_1 for dataset 111
Could not read fold fold_1 for dataset 111
Expected path: /scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset111_AutoPet/autoPET3_Trainer__nnUNetResEncUNetLPlansMultiTalent__3d_fullres/fold_1/test_predictions/summary.json
Error: [Errno 21] Is a directory: '/scratch4/workspace/f007g3j_dartmouth_edu-simple/nnUNet_data/nnUNet_results/Dataset111_AutoPet/autoPET3_Trainer__nnUN

ValueError: No objects to concatenate

In [ ]:

lesion_metrics_df = pd.read_csv('../lesion_metrics.csv')

FileNotFoundError: [Errno 2] No such file or directory: '../lesion_metrics.csv'